In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("SalesDataAnalysis") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 10:29:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df_orders = spark.read.csv("/opt/spark-data/Sales/orders.csv", header=True, inferSchema=True)

In [3]:
df_orders.show(5)

+--------+-----------+----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|   status|
+--------+-----------+----------+----------+--------+----------+---------+
|    1001|       C001|      P101|2024-01-05|       2|     29.99|Completed|
|    1002|       C002|      P102|2024-01-07|       1|    149.99|Completed|
|    1003|       C003|      P103|2024-01-10|       5|      9.99|Completed|
|    1004|       C001|      P104|2024-01-12|       3|     49.99|Cancelled|
|    1005|       C004|      P101|2024-01-15|       1|     29.99|Completed|
+--------+-----------+----------+----------+--------+----------+---------+
only showing top 5 rows


In [4]:
df_cust = spark.read.csv("/opt/spark-data/Sales/customers.csv", header=True, inferSchema=True)

In [5]:
df_cust.show(5)

+-----------+-------------+----------+-------------+---------+------------+
|customer_id|customer_name|      city|       region|  segment|loyalty_tier|
+-----------+-------------+----------+-------------+---------+------------+
|       C001|Alice Johnson|    London|   South East|   Retail|        Gold|
|       C002|    Bob Smith|Manchester|   North West|Wholesale|      Silver|
|       C003| Clara Hughes|Birmingham|West Midlands|   Retail|      Bronze|
|       C004|    David Lee|     Leeds|    Yorkshire|   Retail|        Gold|
|       C005|   Emma Brown|   Bristol|   South West|Wholesale|      Silver|
+-----------+-------------+----------+-------------+---------+------------+
only showing top 5 rows


In [6]:
df_cust.select("region", "segment").show()

+-------------+---------+
|       region|  segment|
+-------------+---------+
|   South East|   Retail|
|   North West|Wholesale|
|West Midlands|   Retail|
|    Yorkshire|   Retail|
|   South West|Wholesale|
|    Yorkshire|   Retail|
|   North West|   Retail|
|East Midlands|Wholesale|
|     Scotland|   Retail|
|        Wales|Wholesale|
+-------------+---------+



In [7]:
df_cust.select(df_cust.region, df_cust.segment).show()

+-------------+---------+
|       region|  segment|
+-------------+---------+
|   South East|   Retail|
|   North West|Wholesale|
|West Midlands|   Retail|
|    Yorkshire|   Retail|
|   South West|Wholesale|
|    Yorkshire|   Retail|
|   North West|   Retail|
|East Midlands|Wholesale|
|     Scotland|   Retail|
|        Wales|Wholesale|
+-------------+---------+



In [8]:
from pyspark.sql.functions import col

In [9]:
df_cust.select(col('region'), col('segment')).show()

+-------------+---------+
|       region|  segment|
+-------------+---------+
|   South East|   Retail|
|   North West|Wholesale|
|West Midlands|   Retail|
|    Yorkshire|   Retail|
|   South West|Wholesale|
|    Yorkshire|   Retail|
|   North West|   Retail|
|East Midlands|Wholesale|
|     Scotland|   Retail|
|        Wales|Wholesale|
+-------------+---------+



In [11]:
df_orders.filter(df_orders.quantity > 4).show()

+--------+-----------+----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|   status|
+--------+-----------+----------+----------+--------+----------+---------+
|    1003|       C003|      P103|2024-01-10|       5|      9.99|Completed|
|    1012|       C008|      P103|2024-02-03|       6|      9.99|Cancelled|
+--------+-----------+----------+----------+--------+----------+---------+



In [12]:
df_orders.filter("quantity > 4").show()

+--------+-----------+----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|   status|
+--------+-----------+----------+----------+--------+----------+---------+
|    1003|       C003|      P103|2024-01-10|       5|      9.99|Completed|
|    1012|       C008|      P103|2024-02-03|       6|      9.99|Cancelled|
+--------+-----------+----------+----------+--------+----------+---------+



In [13]:
df_orders.filter((df_orders.quantity > 4) &
                 (df_orders.status == "Completed")).show() 

+--------+-----------+----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|   status|
+--------+-----------+----------+----------+--------+----------+---------+
|    1003|       C003|      P103|2024-01-10|       5|      9.99|Completed|
+--------+-----------+----------+----------+--------+----------+---------+



In [14]:
df_orders.filter((df_orders.quantity > 4) |
                 (df_orders.status == "Completed")).show() 

+--------+-----------+----------+----------+--------+----------+---------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|   status|
+--------+-----------+----------+----------+--------+----------+---------+
|    1001|       C001|      P101|2024-01-05|       2|     29.99|Completed|
|    1002|       C002|      P102|2024-01-07|       1|    149.99|Completed|
|    1003|       C003|      P103|2024-01-10|       5|      9.99|Completed|
|    1005|       C004|      P101|2024-01-15|       1|     29.99|Completed|
|    1007|       C002|      P103|2024-01-20|       4|      9.99|Completed|
|    1008|       C006|      P102|2024-01-22|       1|    149.99|Completed|
|    1010|       C007|      P105|2024-01-28|       3|     89.99|Completed|
|    1011|       C004|      P101|2024-02-01|       1|     29.99|Completed|
|    1012|       C008|      P103|2024-02-03|       6|      9.99|Cancelled|
|    1013|       C005|      P102|2024-02-06|       2|    149.99|Completed|
|    1014|       C006|   

In [15]:
df_cust.filter(df_cust.region.contains("South")).show()

+-----------+-------------+-------+----------+---------+------------+
|customer_id|customer_name|   city|    region|  segment|loyalty_tier|
+-----------+-------------+-------+----------+---------+------------+
|       C001|Alice Johnson| London|South East|   Retail|        Gold|
|       C005|   Emma Brown|Bristol|South West|Wholesale|      Silver|
+-----------+-------------+-------+----------+---------+------------+



In [16]:
df_cust.withColumnRenamed("segment", "business_segment").show()

+-----------+-------------+----------+-------------+----------------+------------+
|customer_id|customer_name|      city|       region|business_segment|loyalty_tier|
+-----------+-------------+----------+-------------+----------------+------------+
|       C001|Alice Johnson|    London|   South East|          Retail|        Gold|
|       C002|    Bob Smith|Manchester|   North West|       Wholesale|      Silver|
|       C003| Clara Hughes|Birmingham|West Midlands|          Retail|      Bronze|
|       C004|    David Lee|     Leeds|    Yorkshire|          Retail|        Gold|
|       C005|   Emma Brown|   Bristol|   South West|       Wholesale|      Silver|
|       C006| Frank Wilson| Sheffield|    Yorkshire|          Retail|      Bronze|
|       C007|    Grace Kim| Liverpool|   North West|          Retail|        Gold|
|       C008| Henry Taylor|Nottingham|East Midlands|       Wholesale|      Silver|
|       C009|  Isla Martin| Edinburgh|     Scotland|          Retail|        Gold|
|   

In [17]:
df_cust.withColumnRenamed("segment", "business_segment").withColumnRenamed("loyalty_tier", "membership").show()

+-----------+-------------+----------+-------------+----------------+----------+
|customer_id|customer_name|      city|       region|business_segment|membership|
+-----------+-------------+----------+-------------+----------------+----------+
|       C001|Alice Johnson|    London|   South East|          Retail|      Gold|
|       C002|    Bob Smith|Manchester|   North West|       Wholesale|    Silver|
|       C003| Clara Hughes|Birmingham|West Midlands|          Retail|    Bronze|
|       C004|    David Lee|     Leeds|    Yorkshire|          Retail|      Gold|
|       C005|   Emma Brown|   Bristol|   South West|       Wholesale|    Silver|
|       C006| Frank Wilson| Sheffield|    Yorkshire|          Retail|    Bronze|
|       C007|    Grace Kim| Liverpool|   North West|          Retail|      Gold|
|       C008| Henry Taylor|Nottingham|East Midlands|       Wholesale|    Silver|
|       C009|  Isla Martin| Edinburgh|     Scotland|          Retail|      Gold|
|       C010|  James White| 